# Surgical Needle Pipeline: YOLOv8 → SAM3 → Arc Fitting

Full pipeline on a single image:
1. **YOLOv8**: detect the needle bounding box
2. **SAM3**: refine to a pixel-accurate segmentation mask (box + text prompt)
3. **Arc fitting**: fit a circle to the mask foreground to recover the needle's
   geometric center, radius, arc endpoints, and **arc midpoint** (the needle's
   true mid-point along its body).

The arc-fitting stage is robust to **broken masks** (e.g. when specular reflection
leaves the middle of the needle unsegmented): both arc segments still vote for
the same underlying circle, so the fitted circle — and thus the arc midpoint —
remains correct even when the mask is discontinuous.

## Step 0 — Config

Matches the original CLI command:
```
--image        /home/songyu/Datasets/Autosurg/autosurg/output_frames/00202.jpg
--yolo_weights /home/songyu/Project/yolo/YOLOv8_needle/runs/detect/runs/needle/synthetic_v1/weights/best.pt
--conf         0.25
--expand       1
--output       results/autosurg_youtube_demo_single_img/00202_expand1_result.jpg
```

In [ ]:
from pathlib import Path

# ────── Inputs ──────
# IMAGE_PATH   = "/home/songyu/Datasets/Autosurg/autosurg/output_frames/00202.jpg"
IMAGE_PATH   = "/home/songyu/Datasets/Autosurg/cuhk/Cali_Data_Needle_Image/needle_image/left"
YOLO_WEIGHTS = "/home/songyu/Project/yolo/YOLOv8_needle/runs/detect/runs/needle/synthetic_v1/weights/best.pt"

# ────── Detection / segmentation params ──────
YOLO_CONF  = 0.5
BOX_EXPAND = 1.0
SAM3_TEXT  = "suturing needle"

# ────── Output ──────
OUTPUT_PATH = "Project/autosurg/results/cuhk/left"
Path(OUTPUT_PATH).parent.mkdir(parents=True, exist_ok=True)

# ────── Device ──────
import torch
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}")

## Step 1 — YOLOv8 detection + box enlarge

Prior: exactly one needle per image → keep only the highest-confidence detection.

In [ ]:
import numpy as np
from PIL import Image

def detect_and_enlarge(image_path, yolo_weights, conf=0.25, expand=1.0):
    """Run YOLOv8 and return the single highest-confidence enlarged box."""
    from ultralytics import YOLO

    model   = YOLO(yolo_weights)
    results = model(image_path, conf=conf)[0]

    img_h, img_w = results.orig_shape
    boxes_xyxy   = results.boxes.xyxy.cpu().numpy()
    confs        = results.boxes.conf.cpu().numpy()

    if len(boxes_xyxy) == 0:
        print("[WARN] YOLOv8 detected no needles")
        return np.array([]), img_w, img_h

    # Prior: one needle per image — keep only the top-confidence box
    best_idx = confs.argmax()
    if len(boxes_xyxy) > 1:
        print(f"  [INFO] {len(boxes_xyxy)} detections, keeping best "
              f"(idx={best_idx}, conf={confs[best_idx]:.3f})")

    x1, y1, x2, y2 = boxes_xyxy[best_idx]
    cx, cy = (x1 + x2) / 2, (y1 + y2) / 2
    w,  h  = (x2 - x1) * expand, (y2 - y1) * expand

    nx1 = max(0,     cx - w / 2)
    ny1 = max(0,     cy - h / 2)
    nx2 = min(img_w, cx + w / 2)
    ny2 = min(img_h, cy + h / 2)

    print(f"  original: [{x1:.1f}, {y1:.1f}, {x2:.1f}, {y2:.1f}]  conf={confs[best_idx]:.3f}")
    print(f"  enlarged: [{nx1:.1f}, {ny1:.1f}, {nx2:.1f}, {ny2:.1f}]")
    return np.array([[nx1, ny1, nx2, ny2]]), img_w, img_h


boxes_xyxy, img_w, img_h = detect_and_enlarge(
    IMAGE_PATH, YOLO_WEIGHTS, conf=YOLO_CONF, expand=BOX_EXPAND
)
print(f"\nImage: {img_w} x {img_h}")
assert len(boxes_xyxy) > 0, "No needle detected — aborting."

## Step 2 — SAM3 segmentation (box + text prompt)

In [ ]:
def xyxy_to_xywh_norm(box_xyxy, img_w, img_h):
    """Convert absolute XYXY to normalized [cx, cy, w, h] for SAM3."""
    x1, y1, x2, y2 = box_xyxy
    cx = ((x1 + x2) / 2) / img_w
    cy = ((y1 + y2) / 2) / img_h
    w  = (x2 - x1) / img_w
    h  = (y2 - y1) / img_h
    return np.array([cx, cy, w, h], dtype=np.float32)


def sam3_segment(image_path, boxes_xyxy, img_w, img_h, text_prompt=SAM3_TEXT):
    """Run SAM3 on each box; return list of boolean masks and their scores."""
    from sam3.model_builder import build_sam3_image_model
    from sam3.model.sam3_image_processor import Sam3Processor

    # Keep weights + autocast ops in a consistent dtype
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True

    print("[SAM3] Loading model...")
    model     = build_sam3_image_model().to(DEVICE)
    processor = Sam3Processor(model)
    image     = Image.open(image_path).convert("RGB")

    all_masks, all_scores = [], []
    with torch.autocast("cuda", dtype=torch.bfloat16):
        for i, box_xyxy in enumerate(boxes_xyxy):
            print(f"[SAM3] Target {i+1}/{len(boxes_xyxy)}...")
            state  = processor.set_image(image)
            output = processor.set_text_prompt(state=state, prompt=text_prompt)

            box_norm = xyxy_to_xywh_norm(box_xyxy, img_w, img_h)
            output   = processor.add_geometric_prompt(box=box_norm, label=1, state=state)

            # Cast to float32 before .numpy() — numpy does not support bfloat16
            masks  = output["masks"].cpu().float().numpy()
            scores = output["scores"].cpu().float().numpy()

            best_idx = scores.argmax()
            all_masks.append(masks[best_idx, 0].astype(bool))
            all_scores.append(float(scores[best_idx]))
            print(f"  -> mask score: {all_scores[-1]:.3f}")
    return all_masks, all_scores


masks, scores = sam3_segment(IMAGE_PATH, boxes_xyxy, img_w, img_h)
print(f"\nGot {len(masks)} mask(s). Foreground pixel count: {masks[0].sum()}")

## Step 3 — Arc fitting (the new module)

A suturing needle is a **circular arc**. Rather than taking the mask's centroid
(which biases toward the denser arc segment when the mask is broken), we fit a
circle to all foreground points and then find the midpoint along the arc itself.

**Algorithm:**
1. Clean the mask — keep only the largest connected component to drop speckle.
2. Algebraic least-squares circle fit on all remaining foreground pixels.
3. Compute each pixel's angle about the fitted center.
4. The largest gap in the sorted angles corresponds to the needle's open side;
   the two angles bordering that gap are the **arc endpoints**.
5. The **arc midpoint** is the angle halfway between the two endpoints along
   the arc (not across the gap), projected back onto the circle.

In [ ]:
import cv2

def clean_mask(mask, min_area_ratio=0.05):
    """Keep only the largest connected component above a minimum area."""
    m = mask.astype(np.uint8)
    n, labels, stats, _ = cv2.connectedComponentsWithStats(m, connectivity=8)
    if n <= 1:
        return m.astype(bool)
    # Largest non-background component
    areas    = stats[1:, cv2.CC_STAT_AREA]
    largest  = 1 + int(np.argmax(areas))
    cleaned  = (labels == largest)
    # Sanity: if it is too tiny relative to original mask, return original
    if cleaned.sum() < min_area_ratio * m.sum():
        return mask.astype(bool)
    return cleaned


def fit_circle_lsq(xs, ys):
    """Algebraic least-squares circle fit.

    Solve x^2 + y^2 + D*x + E*y + F = 0 for (D, E, F),
    from which center=(-D/2, -E/2) and radius=sqrt(cx^2+cy^2-F).
    """
    xs = xs.astype(np.float64)
    ys = ys.astype(np.float64)
    A  = np.column_stack([xs, ys, np.ones_like(xs)])
    b  = -(xs**2 + ys**2)
    D, E, F = np.linalg.lstsq(A, b, rcond=None)[0]
    cx, cy  = -D / 2, -E / 2
    r2      = cx**2 + cy**2 - F
    if r2 <= 0:
        return None
    return float(cx), float(cy), float(np.sqrt(r2))


def fit_needle_arc(mask):
    """Fit a circle to a (possibly broken) needle mask and locate the arc midpoint.

    Returns a dict with:
        center      : (cx, cy)   fitted circle center (pixel coords)
        radius      : float      fitted circle radius
        endpoints   : ((x1,y1), (x2,y2))   two ends of the arc, on the circle
        midpoint    : (mx, my)   midpoint along the arc, on the circle
        arc_angles  : (a_start, a_end)     endpoint angles in radians
        mid_angle   : float      midpoint angle in radians
        arc_sweep   : float      angular span of the arc in radians
        fit_residual: float      RMS pixel distance from points to fitted circle
        n_points    : int        number of foreground pixels used
    Returns None if fitting failed.
    """
    cleaned = clean_mask(mask)
    ys, xs  = np.where(cleaned)
    if len(xs) < 20:
        print("[ArcFit] too few foreground pixels")
        return None

    fit = fit_circle_lsq(xs, ys)
    if fit is None:
        print("[ArcFit] circle fit failed (non-positive radius)")
        return None
    cx, cy, r = fit

    # Angle of every foreground pixel w.r.t. the fitted center
    angles = np.arctan2(ys - cy, xs - cx)
    sorted_ang = np.sort(angles)

    # Circular gaps between consecutive sorted angles.
    # The LARGEST gap is the 'opening' of the arc; the arc lives across
    # the complementary span.
    wrap = sorted_ang[0] + 2 * np.pi
    gaps = np.diff(np.concatenate([sorted_ang, [wrap]]))
    gap_idx = int(np.argmax(gaps))

    # The two angles bordering the largest gap are the arc endpoints.
    # Arc runs from (gap_idx+1) CCW through to (gap_idx), i.e. NOT across the gap.
    a_end   = float(sorted_ang[gap_idx])                       # gap starts just after this angle
    a_start = float(sorted_ang[(gap_idx + 1) % len(sorted_ang)])  # gap ends just before this

    # Walk from a_start CCW around to a_end — unwrap so a_end > a_start
    a_end_u = a_end if a_end > a_start else a_end + 2 * np.pi
    mid_angle = (a_start + a_end_u) / 2
    arc_sweep = a_end_u - a_start

    # Project back onto the circle
    mx = cx + r * np.cos(mid_angle)
    my = cy + r * np.sin(mid_angle)
    ex1 = cx + r * np.cos(a_start);  ey1 = cy + r * np.sin(a_start)
    ex2 = cx + r * np.cos(a_end);    ey2 = cy + r * np.sin(a_end)

    # Fit quality: RMS radial residual
    radial_dists = np.sqrt((xs - cx) ** 2 + (ys - cy) ** 2)
    fit_residual = float(np.sqrt(np.mean((radial_dists - r) ** 2)))

    return {
        "center":       (cx, cy),
        "radius":       r,
        "endpoints":    ((ex1, ey1), (ex2, ey2)),
        "midpoint":     (mx, my),
        "arc_angles":   (a_start, a_end),
        "mid_angle":    mid_angle,
        "arc_sweep":    arc_sweep,
        "fit_residual": fit_residual,
        "n_points":     int(len(xs)),
    }


arc = fit_needle_arc(masks[0])
if arc is None:
    raise RuntimeError("Arc fitting failed.")

print(f"Fitted center : ({arc['center'][0]:.1f}, {arc['center'][1]:.1f})")
print(f"Radius        : {arc['radius']:.1f} px")
print(f"Arc sweep     : {np.degrees(arc['arc_sweep']):.1f}°")
print(f"Midpoint      : ({arc['midpoint'][0]:.1f}, {arc['midpoint'][1]:.1f})")
print(f"Endpoints     : {arc['endpoints'][0]} -> {arc['endpoints'][1]}")
print(f"Fit RMS       : {arc['fit_residual']:.2f} px  (on {arc['n_points']} pixels)")

### Sanity checks
- `arc_sweep` for a typical suturing needle should be between ~90° and ~270°.
- `fit_residual` should be small relative to `radius` (e.g. < 10–20%); a large
  residual means the mask is noisy or the shape isn't really an arc.
- If the midpoint lands *inside* the arc's hollow rather than on the needle
  body, the gap direction was inverted — revisit the mask quality.

In [ ]:
residual_ratio = arc["fit_residual"] / arc["radius"]
arc_deg = np.degrees(arc["arc_sweep"])

print(f"Residual / radius : {residual_ratio:.3f}  (<0.2 is good)")
print(f"Arc sweep         : {arc_deg:.1f}°        (90°–270° is typical)")

if residual_ratio > 0.25:
    print("⚠️  High residual — mask may be noisy or non-circular.")
if arc_deg < 60 or arc_deg > 300:
    print("⚠️  Unusual arc sweep — check the mask.")

## Step 4 — Visualize & save

Draws: box, mask fill, fitted circle (thin), arc overlay (thick), arc endpoints,
circle center (×), and the **arc midpoint** (filled dot with label).

In [ ]:
def visualize(image_path, boxes_xyxy, masks, scores, arc_info, output_path):
    img     = cv2.imread(image_path)
    overlay = img.copy()

    color_mask   = (0, 255, 100)   # green    — mask fill
    color_box    = (0, 255, 100)
    color_circle = (255, 200, 0)   # cyan-ish — full fitted circle (thin)
    color_arc    = (0, 180, 255)   # orange   — the arc portion (thick)
    color_mid    = (0, 0, 255)     # red      — arc midpoint
    color_end    = (255, 0, 255)   # magenta  — arc endpoints
    color_ctr    = (255, 255, 255) # white    — circle center

    # Mask fill (semi-transparent)
    for m in masks:
        overlay[m] = (overlay[m] * 0.4 + np.array(color_mask) * 0.6).astype(np.uint8)

    # Box + score
    for box, score in zip(boxes_xyxy, scores):
        x1, y1, x2, y2 = map(int, box)
        cv2.rectangle(img, (x1, y1), (x2, y2), color_box, 2)
        cv2.putText(img, f"needle {score:.2f}", (x1, y1 - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_box, 2)

    # Blend mask overlay
    img = cv2.addWeighted(overlay, 0.7, img, 0.3, 0)

    if arc_info is not None:
        cx, cy = arc_info["center"]
        r      = arc_info["radius"]
        mx, my = arc_info["midpoint"]
        (ex1, ey1), (ex2, ey2) = arc_info["endpoints"]
        a_start, a_end = arc_info["arc_angles"]

        # Full circle (thin dashed-ish — OpenCV has no dashed, just thin solid)
        cv2.circle(img, (int(round(cx)), int(round(cy))),
                   int(round(r)), color_circle, 1, lineType=cv2.LINE_AA)

        # Arc (thick). cv2.ellipse uses degrees and clockwise-positive angles
        # in image coords, which matches our atan2 convention.
        a_end_u = a_end if a_end > a_start else a_end + 2 * np.pi
        cv2.ellipse(img,
                    (int(round(cx)), int(round(cy))),
                    (int(round(r)), int(round(r))),
                    0,
                    np.degrees(a_start),
                    np.degrees(a_end_u),
                    color_arc, 3, lineType=cv2.LINE_AA)

        # Circle center (×)
        cxi, cyi = int(round(cx)), int(round(cy))
        cv2.drawMarker(img, (cxi, cyi), color_ctr,
                       markerType=cv2.MARKER_CROSS, markerSize=14, thickness=2)

        # Endpoints
        for (ex, ey) in [(ex1, ey1), (ex2, ey2)]:
            cv2.circle(img, (int(round(ex)), int(round(ey))), 5,
                       color_end, -1, lineType=cv2.LINE_AA)

        # Arc midpoint — big filled dot + label
        mxi, myi = int(round(mx)), int(round(my))
        cv2.circle(img, (mxi, myi), 7, color_mid, -1, lineType=cv2.LINE_AA)
        cv2.circle(img, (mxi, myi), 9, (255, 255, 255), 2, lineType=cv2.LINE_AA)
        cv2.putText(img, "mid", (mxi + 10, myi - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.6, color_mid, 2)

    Path(output_path).parent.mkdir(parents=True, exist_ok=True)
    cv2.imwrite(output_path, img)
    print(f"[✓] Saved: {output_path}")
    return img


result_img = visualize(IMAGE_PATH, boxes_xyxy, masks, scores, arc, OUTPUT_PATH)

# Quick inline preview
import matplotlib.pyplot as plt
plt.figure(figsize=(10, 7))
plt.imshow(cv2.cvtColor(result_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.title("YOLO → SAM3 → Arc fitting")
plt.tight_layout()
plt.show()

## Step 5 — Downstream output

The arc-fitting result is the actionable output for downstream robotics
(grasp planning, needle driving). Package it:

In [ ]:
result = {
    "image":        IMAGE_PATH,
    "box_xyxy":     boxes_xyxy[0].tolist(),
    "mask_score":   scores[0],
    "arc_center":   arc["center"],       # circle center in pixels
    "arc_radius":   arc["radius"],       # circle radius in pixels
    "arc_midpoint": arc["midpoint"],     # needle midpoint on the arc
    "arc_endpoints": arc["endpoints"],   # two tips of the visible arc
    "arc_sweep_deg": float(np.degrees(arc["arc_sweep"])),
    "fit_residual":  arc["fit_residual"],
}
for k, v in result.items():
    print(f"{k:15s}: {v}")